In [ ]:
import json
from pathlib import Path


In [ ]:
def load_probe_results(path):
    """Load probe results from a run's summary.json.

    Args:
        path: Path to a run's ``summary.json`` **or** the run's ``output_dir``
            (in which case ``summary.json`` is resolved inside it).

    Returns:
        A flat dict combining run identity with the run's probe metrics, suitable
        for ``pd.DataFrame([...])``. Probe metrics are the summary's ``final_probe_*``
        fields with that prefix stripped, so ``final_probe_score`` -> ``score``
        (the headline mean_probe_score), ``final_probe_linear_mean_f1`` ->
        ``linear_mean_f1``, etc. ``probe_completed`` is False (and no metric keys
        are present) when the run has no probe result (probes disabled/unfinished).

    Raises:
        FileNotFoundError: if summary.json does not exist.
    """
    path = Path(path)
    summary_path = path / "summary.json" if path.is_dir() else path
    if not summary_path.exists():
        raise FileNotFoundError(f"no summary.json at {summary_path}")
    summary = json.loads(summary_path.read_text())

    record = {
        "output_dir": str(summary_path.parent),
        "project": summary.get("project"),
        "family": summary.get("family"),
        "recipe_id": summary.get("recipe_id"),
        "wandb_url": (summary.get("wandb") or {}).get("url"),
        "stop_reason": summary.get("stop_reason"),
        "steps_completed": summary.get("steps_completed"),
        "train_flops": summary.get("train_flops"),
        "flop_fraction": summary.get("flop_fraction"),
    }

    probe = {key.removeprefix("final_probe_"): value
             for key, value in summary.items() if key.startswith("final_probe_")}
    record["probe_completed"] = "score" in probe
    record.update(probe)
    return record

In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

# Ordered {summary key -> display label}. Keys are top-level in summary.json.
PROBE_METRICS = {
    "final_probe_linear_mean_f1":       "Linear F1",
    "final_probe_knn_mean_f1":          "kNN F1",
    "final_probe_fewshot_mean_f1":      "Few-shot F1",
    "final_probe_slide_mean_auc":       "Slide AUC",
    "final_probe_seg_mean_jaccard":     "Seg Jaccard",
    "final_probe_auc_mean":             "AUC",
    "final_probe_survival_mean_cindex": "Survival C-index",
    "final_probe_robustness_mean":      "Robustness",
    "final_probe_score":                "Overall score",
}


def _as_summary(run):
    """Accept a summary dict, a path to summary.json, or a run output_dir."""
    if isinstance(run, dict):
        return run
    path = Path(run)
    path = path / "summary.json" if path.is_dir() else path
    return json.loads(path.read_text())


def plot_probe_comparison(runs, metrics=PROBE_METRICS, ncols=3, figsize=None):
    """Bar-plot probe metrics across N runs, one subplot per metric.

    Args:
        runs: mapping {run_name: summary}, where `summary` is a dict with the
            top-level `final_probe_*` keys, a path to summary.json, or an output_dir.
        metrics: ordered {summary_key: display_label} to plot (one subplot each).
        ncols: number of columns in the subplot grid.
        figsize: optional (w, h); defaults to scale with the grid.

    Returns:
        (fig, axes).
    """
    names = list(runs)
    summaries = {name: _as_summary(run) for name, run in runs.items()}

    n = len(metrics)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize or (4 * ncols, 3 * nrows),
                             squeeze=False)
    palette = sns.color_palette("colorblind", len(names))

    for ax, (key, label) in zip(axes.flat, metrics.items()):
        values = [summaries[name].get(key, float("nan")) for name in names]
        sns.barplot(x=names, y=values, hue=names, palette=palette, legend=False, ax=ax)
        ax.set_title(label)
        ax.set_ylabel("")
        ax.set_ylim(0, 1)                       # all metrics are bounded scores in [0, 1]
        ax.tick_params(axis="x", rotation=45)
        for container in ax.containers:
            ax.bar_label(container, fmt="%.3f", fontsize=8, padding=2)

    for ax in axes.flat[n:]:                    # hide unused grid cells
        ax.set_visible(False)

    fig.tight_layout()
    return fig, axes